In [1]:
import numpy as np
import pandas as pd
import cv2
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.models import load_model

In [2]:
import cv2
import os

img = cv2.imread("../modelo_yolo/imagen_recortada.png")
alto, ancho = img.shape[:2]

cell_h = alto // 9
cell_w = ancho // 9

os.makedirs("celdas", exist_ok=True)
contador = 0

# Margen dinámico o más pequeño para evitar celdas vacías o negativas
margen_h = max(2, cell_h // 12)
margen_w = max(2, cell_w // 12)

for fila in range(9):
    for columna in range(9):
        y1 = fila * cell_h
        y2 = (fila + 1) * cell_h
        x1 = columna * cell_w
        x2 = (columna + 1) * cell_w

        celda = img[y1 + margen_h : y2 - margen_h, x1 + margen_w : x2 - margen_w]
        cv2.imwrite(f"celdas/celda_{contador}.jpg", celda)
        contador += 1

print(f"{contador} celdas guardadas con éxito.")

81 celdas guardadas con éxito.


In [3]:
import tensorflow as tf
import numpy as np

# Cargar MNIST
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = tf.keras.datasets.mnist.load_data()

# Eliminar el dígito 0 original de MNIST para evitar colisiones
mask_train = y_train_raw != 0
mask_test = y_test_raw != 0

X_train = X_train_raw[mask_train]
y_train = y_train_raw[mask_train]
X_test = X_test_raw[mask_test]
y_test = y_test_raw[mask_test]

# 2. Crear imágenes completamente vacías (casilla vacía = clase 0)
imagenes_vacias = np.zeros((5000, 28, 28), dtype=np.uint8)
etiquetas_vacias = np.zeros(5000, dtype=np.uint8)

# Concatenar las nuevas imágenes vacías
X_train = np.concatenate([X_train, imagenes_vacias], axis=0)
y_train = np.concatenate([y_train, etiquetas_vacias], axis=0)

# 3. BARAJAR (Shuffle) manualmente antes del fit para corregir el problema de val_accuracy
indices = np.arange(X_train.shape[0])
np.random.shuffle(indices)
X_train = X_train[indices]
y_train = y_train[indices]

# Normalizar y redimensionar para la CNN
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

In [4]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential([
    # Usamos Input(shape) según la advertencia de Keras
    tf.keras.layers.Input(shape=(28, 28, 1)),
    Conv2D(32, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(10, activation="softmax") # 10 clases (del 0 al 9)
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Ahora verás cómo la val_accuracy sube a más del 98% a la par que el entrenamiento
model.fit(
    X_train,
    y_train,
    epochs=10,
    validation_split=0.2,
    batch_size=64
)

# ¡CRUCIAL! Guardar el modelo para poder cargarlo después
model.save("modelo2_cnn.keras")
print("Modelo guardado correctamente.")

Epoch 1/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 18s 22ms/step - accuracy: 0.9469 - loss: 0.1790 - val_accuracy: 0.9807 - val_loss: 0.0609
Epoch 2/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 19s 26ms/step - accuracy: 0.9845 - loss: 0.0509 - val_accuracy: 0.9865 - val_loss: 0.0432
Epoch 3/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 18s 24ms/step - accuracy: 0.9886 - loss: 0.0357 - val_accuracy: 0.9876 - val_loss: 0.0400
Epoch 4/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - accuracy: 0.9921 - loss: 0.0253 - val_accuracy: 0.9892 - val_loss: 0.0336
Epoch 5/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 16s 21ms/step - accuracy: 0.9931 - loss: 0.0202 - val_accuracy: 0.9894 - val_loss: 0.0381
Epoch 6/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.9954 - loss: 0.0150 - val_accuracy: 0.9882 - val_loss: 0.0400
Epoch 7/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 17s 22ms/step - accuracy: 0.9954 - loss: 0.0137 - val_accuracy: 0.9893 - val_loss: 0.0366
Epoch 8/10
739/739 ━━━━━━━━━━━━━━━━━━━━ 16s 22ms/step - accuracy: 0.9958 - loss: 0.0118 - 

In [ ]:
import cv2
import numpy as np
from tensorflow.keras.models import load_model

# 1. Cargar el modelo entrenado con MNIST
model = load_model("modelo2_cnn.keras")
tablero = []

for i in range(81):
    # 2. Leer la celda guardada en escala de grises
    img = cv2.imread(f"celdas/celda_{i}.jpg", cv2.IMREAD_GRAYSCALE)
    
    # -------------------------------------------------------------------------
    # EXTRACCIÓN Y LIMPIEZA ADAPTATIVA DE CONTORNOS (Anti-descentrado de YOLO)
    # -------------------------------------------------------------------------
    # Binarizamos temporalmente la celda completa para buscar elementos
    _, img_init_bin = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Buscamos todos los contornos presentes en la celda
    contornos_padre, _ = cv2.findContours(img_init_bin, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    
    img_final = np.zeros((28, 28), dtype=np.uint8)
    encontrado = False
    w_box, h_box = 0, 0
    
    if contornos_padre:
        h_c, w_c = img.shape
        contornos_validos = []
        
        for c in contornos_padre:
            x, y, w_box_c, h_box_c = cv2.boundingRect(c)
            
            # Filtro: Si el contorno toca los bordes extremos del recorte y es gigante,
            # significa que es la línea de la cuadrícula azul/negra desalineada. Lo ignoramos.
            if x <= 1 or y <= 1 or (x + w_box_c) >= w_c - 1 or (y + h_box_c) >= h_c - 1:
                if cv2.contourArea(c) > (w_c * h_c * 0.3):
                    continue
            contornos_validos.append(c)
            
        if contornos_validos:
            # El contorno más grande de los válidos será nuestro número intacto
            c_numero = max(contornos_validos, key=cv2.contourArea)
            x, y, w_box, h_box = cv2.boundingRect(c_numero)
            
            # Si el tamaño mínimo es razonable, procedemos a extraerlo
            if w_box >= 3 and h_box >= 6:
                digito = img_init_bin[y:y+h_box, x:x+w_box]
                
                # Redimensionamos el número dándole el colchón negro simétrico de MNIST
                factor = min(18 / h_box, 18 / w_box)
                nuevo_w = int(w_box * factor)
                nuevo_h = int(h_box * factor)
                digito_scaled = cv2.resize(digito, (nuevo_w, nuevo_h))
                
                # Centramos el dígito escalado en el lienzo de 28x28
                start_y = (28 - nuevo_h) // 2
                start_x = (28 - nuevo_w) // 2
                img_final[start_y:start_y+nuevo_h, start_x:start_x+nuevo_w] = digito_scaled
                encontrado = True

    # Si no se encontró ningún número o quedó vacío tras el filtro, añadimos un 0 (Vacío)
    if not encontrado:
        tablero.append(0)
        continue
        
    # Control extra: Si el número de píxeles final es ridículo, es una celda vacía
    if cv2.countNonZero(img_final) < 20:
        tablero.append(0)
        continue

    # -------------------------------------------------------------------------
    # PROCESAMIENTO Y PREDICCIÓN CON LA CNN
    # -------------------------------------------------------------------------
    img_cnn = img_final.astype("float32") / 255.0
    img_cnn = img_cnn.reshape(1, 28, 28, 1)
    
    pred = model.predict(img_cnn, verbose=0)
    clase_predicha = np.argmax(pred)
    
    # -------------------------------------------------------------------------
    # FILTRO GEOMÉTRICO (Doble verificación para desempate del 7/1)
    # -------------------------------------------------------------------------
    # Si la red insiste en que es un 1, miramos la relación de aspecto del contorno original.
    # Un '1' tipográfico real es muy estrecho. El '7' siempre conserva más anchura arriba.
    if clase_predicha == 1 and encontrado:
        relacion_aspecto = w_box / float(h_box)
        if relacion_aspecto > 0.48:
            clase_predicha = 7
            
    tablero.append(int(clase_predicha))

# 8. Reconstruir la matriz final de 9x9 limpia
sudoku_matriz = np.array(tablero).reshape(9, 9)
print("\nMatriz del Sudoku generada con éxito:")
print(sudoku_matriz)



print("¡Matriz exportada con éxito para el Modelo 3!")
# --- GUARDAR LA MATRIZ PARA EL MODELO 3 ---
# Opción A: Guardarlo en formato binario de Numpy (.npy) -> Es la más limpia
np.save("sudoku_detectado.npy", sudoku_matriz)

# Opción B (Opcional): Guardarlo como un archivo de texto común (.txt) por si quieres abrirlo a mano y verlo
np.savetxt("sudoku_detectado.txt", sudoku_matriz, fmt="%d")


Matriz del Sudoku generada con éxito:
[[0 0 2 7 0 6 4 9 3]
 [5 0 0 0 0 0 6 0 0]
 [0 0 7 0 2 0 5 8 0]
 [0 0 5 6 3 0 8 0 0]
 [0 0 3 0 0 5 0 0 2]
 [4 7 8 0 0 9 0 0 0]
 [0 5 0 0 0 0 2 0 6]
 [3 0 0 0 4 0 0 0 0]
 [0 0 0 3 6 0 7 0 0]]
¡Matriz exportada con éxito para el Modelo 3!


In [6]:
# --- GUARDAR LA MATRIZ PARA EL MODELO 3 ---
# Opción A: Guardarlo en formato binario de Numpy (.npy) -> Es la más limpia
np.save("sudoku_detectado.npy", sudoku_matriz)

# Opción B (Opcional): Guardarlo como un archivo de texto común (.txt) por si quieres abrirlo a mano y verlo
np.savetxt("sudoku_detectado.txt", sudoku_matriz, fmt="%d")